Lab 02 — Static scan & policy (Module 2).

Scan malicious vs benign MCP tool definitions and print WARDEN findings.

Run:  python labs/lab02_static_scan.py
      COURSE_LANG=ru python labs/lab02_static_scan.py
Exercise: python labs/run_exercises.py --module m2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexar76/aimarket-courses/blob/main/mcp-security-course/notebooks/{})


In [ ]:
# Setup — run this cell once per session
# Live oracle sandbox (clones alexar76/oracles) — run once per Colab session
import os
import subprocess
import sys

REPO = "https://github.com/alexar76/aimarket-courses.git"
DEST = "/content/aimarket-courses"

if not os.path.isdir(DEST):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, DEST], check=True)
WORKDIR = os.path.join(DEST, "mcp-security-course")
os.chdir(WORKDIR)

# Live oracle sandbox — clone the AIMarket oracles this course calls, then install them.
if not os.path.isdir("_deps/oracles"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/alexar76/oracles.git", "_deps/oracles"], check=True)
for _pkg in ["core", "oracles/lumen"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"_deps/oracles/{_pkg}"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[oracles,dev]"], check=True)
os.environ.setdefault("COURSE_LANG", "en")  # change to ru or es


In [ ]:
"""Lab 02 — Static scan & policy (Module 2).

Scan malicious vs benign MCP tool definitions and print WARDEN findings.

Run:  python labs/lab02_static_scan.py
      COURSE_LANG=ru python labs/lab02_static_scan.py
Exercise: python labs/run_exercises.py --module m2
"""

from __future__ import annotations



from courselib.i18n import get_translator
from courselib.trace import Trace
from courselib.warden import (
    BENIGN_SERVER,
    BENIGN_TOOLS,
    MALICIOUS_TOOLS,
    StaticScanGate,
    WardenGateInput,
    WardenPolicy,
)


def _scan(label: str, tools, trace: Trace) -> None:
    gate = StaticScanGate()
    result = gate.evaluate(
        WardenGateInput(
            server=BENIGN_SERVER,
            tools=tools,
            prior=[],
            policy=WardenPolicy(),
        )
    )
    print(f"\n--- {label} ---")
    print(f"score: {result.score:.2f}  findings: {len(result.findings)}")
    for f in result.findings:
        print(f"  [{f.severity}] {f.code} — {f.message}")
        trace.log("finding", gate=f.gate, code=f.code, severity=f.severity, tool=f.tool)


def main() -> None:
    t = get_translator()
    trace = Trace()
    print(f"== {t('modules.m2.title')} ==")
    print(t("modules.m2.concept"))
    _scan(t("labs.lab02.benign_label"), BENIGN_TOOLS, trace)
    _scan(t("labs.lab02.malicious_label"), MALICIOUS_TOOLS, trace)
    print(f"\n{t('ui.trace')} ({len(trace)} {t('ui.events')}):")
    for event in trace.of_kind("finding"):
        print(" ", event["code"], event.get("severity"), event.get("tool"))
    print(f"\n--- {t('exercises.heading')} ---")
    print(t("exercises.lab02_hint"))

main()
